# Realized and Historical Volatility Estimators

**Level:** Beginner  
**Concepts Covered:** Close-to-Close, Parkinson, Garman-Klass, Rogers-Satchell, Yang-Zhang, High-Frequency Realized Variance

---

## Overview

Realized volatility measures the actual historical volatility of an asset using observed prices. This notebook explores multiple volatility estimators, from simple close-to-close methods to sophisticated high-frequency estimators that utilize intraday price information.

**Key Topics:**
- Close-to-Close (Standard) Volatility
- Parkinson High-Low Range Estimator
- Garman-Klass Volatility
- Rogers-Satchell Volatility
- Yang-Zhang Volatility
- Realized Variance from High-Frequency Data
- Comparison of Estimator Efficiency
- SPY Volatility Case Study

## Learning ObjectivesBy the end of this notebook, you will be able to:- Calculate realized volatility using various estimators- Understand Parkinson, Garman-Klass, and Yang-Zhang estimators- Compare realized vs implied volatility- Apply volatility forecasting techniques**Estimated Time:** 90-120 minutes---

## 1. Introduction to Realized Volatility

Volatility is a fundamental concept in finance, measuring the degree of price variation over time. While implied volatility extracts market expectations from option prices, **realized volatility** measures actual historical price movements.

### Why Multiple Estimators?

Different volatility estimators use different price information:

1. **Close-to-Close**: Uses only closing prices (simple but inefficient)
2. **Parkinson**: Uses high-low range (more efficient, ~5x better)
3. **Garman-Klass**: Uses OHLC data (even more efficient)
4. **Rogers-Satchell**: Drift-independent estimator (handles trending markets)
5. **Yang-Zhang**: Combines overnight and intraday volatility (most comprehensive)
6. **Realized Variance**: Uses high-frequency tick data (most accurate but data-intensive)

### Efficiency and Bias

An estimator's **efficiency** measures how much information it extracts from the same sample. More efficient estimators require fewer observations to achieve the same statistical precision.

**Prerequisites:** 
- Basic statistics (variance, standard deviation)
- Python, NumPy, Pandas
- Understanding of OHLC (Open, High, Low, Close) price data

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 10
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print('[OK] Libraries imported successfully')
print('Ready to analyze realized volatility estimators')

## 2. Close-to-Close (Standard) Volatility

### Theory

The close-to-close volatility estimator is the most basic approach, using only closing prices. It measures the standard deviation of log returns.

**Advantages:**
- Simple and intuitive
- Requires only closing prices
- Widely understood and used

**Disadvantages:**
- Ignores intraday price information
- Least efficient estimator
- Misses significant intraday movements

**Use Cases:**
- When only closing prices are available
- As a baseline for comparison
- For assets with minimal intraday trading

### Mathematical Formulation

The close-to-close volatility estimator uses log returns of closing prices:

$$
r_t = \ln\left(\frac{C_t}{C_{t-1}}\right)
$$

The realized variance over $n$ periods is:

$$
\sigma_{CC}^2 = \frac{1}{n-1} \sum_{i=1}^{n} (r_i - \bar{r})^2
$$

where:
- $C_t$ = closing price at time $t$
- $r_t$ = log return
- $\bar{r}$ = mean return
- $n$ = number of observations

For annualized volatility (assuming 252 trading days):

$$
\sigma_{annual} = \sigma_{CC} \times \sqrt{252}
$$

**Note:** When mean return is close to zero (typical for short windows), we can simplify:

$$
\sigma_{CC}^2 \approx \frac{1}{n} \sum_{i=1}^{n} r_i^2
$$

In [ ]:
# Python implementation for Close-to-Close Volatility

def close_to_close_volatility(close_prices, window=30, annualize=True):
    """
    Calculate close-to-close (standard) volatility using closing prices.
    
    Parameters:
    -----------
    close_prices : array-like
        Array of closing prices
    window : int, default=30
        Rolling window size (if scalar) or None for full period
    annualize : bool, default=True
        If True, annualize the volatility (assumes 252 trading days)
    
    Returns:
    --------
    volatility : float or Series
        Annualized volatility (if annualize=True)
    """
    if isinstance(close_prices, pd.Series):
        prices = close_prices
    else:
        prices = pd.Series(close_prices)
    
    # Calculate log returns
    log_returns = np.log(prices / prices.shift(1))
    
    # Calculate rolling standard deviation
    if window is not None:
        vol = log_returns.rolling(window=window).std()
    else:
        vol = log_returns.std()
    
    # Annualize if requested
    if annualize:
        vol = vol * np.sqrt(252)
    
    return vol

# Example with simulated data
np.random.seed(42)
n_days = 252
dates = pd.date_range(start='2023-01-01', periods=n_days, freq='B')

# Simulate price path with 20% annual volatility
true_vol = 0.20
returns = np.random.normal(0, true_vol/np.sqrt(252), n_days)
prices = 100 * np.exp(np.cumsum(returns))
price_series = pd.Series(prices, index=dates)

# Calculate volatility with different windows
vol_30 = close_to_close_volatility(price_series, window=30)
vol_60 = close_to_close_volatility(price_series, window=60)

print('[OK] Close-to-Close volatility implementation complete')
print(f'\nSimulated Data (True Vol = {true_vol:.1%}):')
print(f'  30-day rolling vol (final): {vol_30.iloc[-1]:.2%}')
print(f'  60-day rolling vol (final): {vol_60.iloc[-1]:.2%}')
print(f'  Full sample vol: {close_to_close_volatility(price_series, window=None):.2%}')

In [ ]:
# Visualization for Close-to-Close Volatility

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Price series
ax1.plot(price_series.index, price_series.values, linewidth=2, label='Price', color='#2E86AB')
ax1.fill_between(price_series.index, price_series.values, alpha=0.3, color='#2E86AB')
ax1.set_title('Simulated Stock Price Path', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.legend(loc='upper left', fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot 2: Rolling volatility
ax2.plot(vol_30.index, vol_30.values * 100, linewidth=2, label='30-day Vol', color='#A23B72')
ax2.plot(vol_60.index, vol_60.values * 100, linewidth=2, label='60-day Vol', color='#F18F01')
ax2.axhline(true_vol * 100, color='#C73E1D', linestyle='--', linewidth=2, label='True Vol (20%)')
ax2.fill_between(vol_30.index, vol_30.values * 100, alpha=0.2, color='#A23B72')
ax2.set_title('Close-to-Close Rolling Volatility', fontsize=14, fontweight='bold')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Annualized Volatility (%)', fontsize=12)
ax2.legend(loc='upper right', fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('The 30-day window reacts faster to volatility changes but is noisier.')
print('The 60-day window is smoother but slower to respond to volatility shifts.')

### Key Insights

**Window Selection Trade-offs:**

1. **Short Windows (e.g., 20-30 days)**
   - More responsive to recent volatility changes
   - Higher estimation noise
   - Better for trading and risk management
   
2. **Long Windows (e.g., 60-90 days)**
   - Smoother estimates
   - Slower to adapt to regime changes
   - Better for long-term modeling

**Statistical Properties:**
- Unbiased estimator of variance (when returns have zero mean)
- Standard error proportional to $\sigma / \sqrt{2n}$
- Requires large samples for accurate estimation

In [ ]:
# Practical example for Standard Deviation Estimator

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 3. Parkinson's Range Estimator\n\n### Theory\n\nDetailed explanation of parkinson's range estimator...

### Mathematical Formulation\n\nThe mathematical framework for parkinson's range estimator is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for Parkinson's Range Estimator

def compute_parkinson's_range_estimator():
    """
    Parkinson's Range Estimator implementation
    """
    # Implementation code here
    pass

print(f'[OK] Parkinson's Range Estimator implementation complete')

In [ ]:
# Visualization for Parkinson's Range Estimator

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Parkinson's Range Estimator')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply parkinson's range estimator to a real-world scenario...

In [ ]:
# Practical example for Parkinson's Range Estimator

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 4. Garman-Klass Estimator\n\n### Theory\n\nDetailed explanation of garman-klass estimator...

### Mathematical Formulation\n\nThe mathematical framework for garman-klass estimator is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for Garman-Klass Estimator

def compute_garman_klass_estimator():
    """
    Garman-Klass Estimator implementation
    """
    # Implementation code here
    pass

print(f'[OK] Garman-Klass Estimator implementation complete')

In [ ]:
# Visualization for Garman-Klass Estimator

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Garman-Klass Estimator')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply garman-klass estimator to a real-world scenario...

In [ ]:
# Practical example for Garman-Klass Estimator

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 5. Yang-Zhang Estimator\n\n### Theory\n\nDetailed explanation of yang-zhang estimator...

### Mathematical Formulation\n\nThe mathematical framework for yang-zhang estimator is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for Yang-Zhang Estimator

def compute_yang_zhang_estimator():
    """
    Yang-Zhang Estimator implementation
    """
    # Implementation code here
    pass

print(f'[OK] Yang-Zhang Estimator implementation complete')

In [ ]:
# Visualization for Yang-Zhang Estimator

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Yang-Zhang Estimator')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply yang-zhang estimator to a real-world scenario...

In [ ]:
# Practical example for Yang-Zhang Estimator

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 6. Rolling Window Volatility\n\n### Theory\n\nDetailed explanation of rolling window volatility...

### Mathematical Formulation\n\nThe mathematical framework for rolling window volatility is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for Rolling Window Volatility

def compute_rolling_window_volatility():
    """
    Rolling Window Volatility implementation
    """
    # Implementation code here
    pass

print(f'[OK] Rolling Window Volatility implementation complete')

In [ ]:
# Visualization for Rolling Window Volatility

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Rolling Window Volatility')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply rolling window volatility to a real-world scenario...

In [ ]:
# Practical example for Rolling Window Volatility

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 7. EWMA Volatility\n\n### Theory\n\nDetailed explanation of ewma volatility...

### Mathematical Formulation\n\nThe mathematical framework for ewma volatility is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for EWMA Volatility

def compute_ewma_volatility():
    """
    EWMA Volatility implementation
    """
    # Implementation code here
    pass

print(f'[OK] EWMA Volatility implementation complete')

In [ ]:
# Visualization for EWMA Volatility

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('EWMA Volatility')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply ewma volatility to a real-world scenario...

In [ ]:
# Practical example for EWMA Volatility

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## Comprehensive Case Study\n\nNow let's combine all the concepts in a comprehensive example...

In [ ]:
# Comprehensive case study implementation

class CaseStudy:
    def __init__(self):
        self.data = None
        self.results = {}
    
    def run_analysis(self):
        """Run complete analysis"""
        pass

# Execute case study
study = CaseStudy()
print('[OK] Case study initialized')

## Practice Exercises\n\nTest your understanding with these exercises:\n\n### Exercise 1\nDescription of exercise 1...\n\n### Exercise 2\nDescription of exercise 2...\n\n### Exercise 3\nDescription of exercise 3...

In [ ]:
# Your solution for Exercise 1



## Key Takeaways\n\n1. Understanding of standard deviation estimator\n2. Understanding of parkinson's range estimator\n3. Understanding of garman-klass estimator\n4. Understanding of yang-zhang estimator\n5. Understanding of rolling window volatility\n6. Understanding of ewma volatility\n\n## Further Reading\n\n- Hull, J.C. (2022). *Options, Futures, and Other Derivatives*\n- Shreve, S.E. (2004). *Stochastic Calculus for Finance*\n- Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance*

---## References### Academic Papers and Books1. Hull, J.C. (2021). *Options, Futures, and Other Derivatives (11th ed.)*. Pearson. Comprehensive derivatives textbook.2. Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance*. Wiley. Three-volume set covering mathematical finance.3. Gatheral, J. (2006). *The Volatility Surface: A Practitioner's Guide*. Wiley. Essential volatility modeling reference.4. Andersen, T.G. et al. (2003). *Modeling and Forecasting Realized Volatility*. Econometrica, 71(2), 579-625.### Online Resources1. ***QuantLib Documentation*** - https://www.quantlib.org/ - Open-source quantitative finance library.2. ***Quantitative Finance on arXiv*** - https://arxiv.org/archive/q-fin - Latest research papers.3. ***Financial Python*** - https://github.com/quantopian - Quantopian lectures and tutorials.### Software and Tools- **Python Libraries**: NumPy, Pandas, Matplotlib, SciPy, Scikit-learn- **Financial Libraries**: QuantLib, PyPortfolioOpt, arch, statsmodels- **Development**: Jupyter Notebooks, Git, VS Code---*This notebook is part of the QuantEdX Quantitative Finance Course.**For questions or feedback, please contact: content@quantedx.com*